# NeoOLAF native EventStoryLine layer ablation — one document v1.5

This v1.5 notebook keeps the same **OWL-Time** ontology used in RAGTree and the controlled labels `PRECONDITION`/`FALLING_ACTION`. It focuses on the relation bottleneck without modifying `src/neoolaf`:

- parallel sentence event extraction plus **three** whole-document coverage reviews;
- deterministic adaptive **unordered pair coverage** with zero relation-generation LLM calls;
- atomic verified pair-local relation decisions that jointly choose existence, direction, and class;
- exact local context, schema-exact positive/negative/reversed examples, evidence requirements, recovery, caching, and `NONE` filtering before Layer 3;
- exact-span-only projected metrics plus strict native span metrics, candidate-pool recall, confusion matrix, and detailed first-failure tracing.

Gold remains unavailable until after Layer 12.


In [5]:
from __future__ import annotations

import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


def first_existing_path(env_name: str, candidates: list[Path]) -> Path:
    raw = os.environ.get(env_name, "").strip().strip('"').strip("'")
    if raw:
        path = Path(raw).expanduser().resolve()
        if path.is_file():
            return path
        raise FileNotFoundError(f"{env_name} points to a missing file: {path}")
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {env_name}. Tried:\n"
        + "\n".join(str(Path(x).expanduser().resolve()) for x in candidates)
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from eventstoryline_native_ablation_v1_5 import (
    RELATION_IDS,
    analyze_run,
    gold_event_index,
    indexed_token_table,
    load_layer_states,
    project_event_label,
    read_json,
    read_jsonl,
    run_native_pipeline,
    seed_ontology_summary,
)

print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = /home/galencarmedeiro/git/postdoc/NeoOLAF


## 1. Configuration

In [6]:
INPUT_JSONL = NOTEBOOK_DIR / "data/eventstoryline_one_input_v1.jsonl"
GOLD_JSONL = NOTEBOOK_DIR / "data/eventstoryline_one_gold_v1.jsonl"
SMOKE5_INPUT_JSONL = NOTEBOOK_DIR / "data/eventstoryline_smoke5_input_v1.jsonl"
SMOKE5_GOLD_JSONL = NOTEBOOK_DIR / "data/eventstoryline_smoke5_gold_v1.jsonl"

# Same seed ontology used by the RAGTree EventStoryLine experiments.
# Override explicitly with EVENTSTORYLINE_ONTOLOGY_PATH when needed.
ONTOLOGY_PATH = first_existing_path(
    "EVENTSTORYLINE_ONTOLOGY_PATH",
    [
        PROJECT_ROOT.parent / "ragtree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT.parent / "RAGTree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT / "../ragtree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT / "../RAGTree/data/ontology/OWLTime/time.ttl",
    ],
)

# Controlled normalized benchmark relation schema. This is not the seed ontology.
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/eventstoryline_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/eventstoryline_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_profile_native_ablation_v1_5.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_eventstoryline_native_ablation_v1_5.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_task_guidance_v1_5.json"

RUNS_ROOT = NOTEBOOK_DIR / "runs/eventstoryline_native_layer_ablation"
RUN_DIR = RUNS_ROOT / "document_1_10ecbplus_v1_5_owltime_atomic_verified"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Pair batches are independent and run concurrently in Layer 2.
WORKERS = 16
REASONING_EFFORT = "minimal"
RUN_PIPELINE = True
CLEAN_RUN_DIR = True

print("Input:", INPUT_JSONL)
print("Gold:", GOLD_JSONL)
print("OWL-Time seed ontology:", ONTOLOGY_PATH)
print("Controlled relation catalog:", RELATION_CATALOG)
print("Profile:", PROFILE_PATH)
print("Guidance:", GUIDANCE_PATH)
print("Task guidance:", TASK_GUIDANCE_PATH)
print("Run dir:", RUN_DIR)
print("Model:", MODEL_NAME)
print("API key available:", bool(API_KEY))


Input: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/data/eventstoryline_one_input_v1.jsonl
Gold: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/data/eventstoryline_one_gold_v1.jsonl
OWL-Time seed ontology: /home/galencarmedeiro/git/postdoc/ragtree/data/ontology/OWLTime/time.ttl
Controlled relation catalog: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/ontology/eventstoryline_relation_catalog.json
Profile: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/eventstoryline_profile_native_ablation_v1_5.json
Guidance: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/guidance_eventstoryline_native_ablation_v1_5.json
Task guidance: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/configs/eventstoryline_task_guidance_v1_5.json
Run dir: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v

## 2. Preflight: anti-leakage, OWL-Time, deterministic pair coverage, five-document files, and event identity


In [7]:
required = [
    INPUT_JSONL, GOLD_JSONL, SMOKE5_INPUT_JSONL, SMOKE5_GOLD_JSONL,
    ONTOLOGY_PATH, RELATION_CATALOG, RELATION_ALIASES,
    PROFILE_PATH, GUIDANCE_PATH, TASK_GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

input_rows = read_jsonl(INPUT_JSONL)
gold_rows = read_jsonl(GOLD_JSONL)
smoke_input_rows = read_jsonl(SMOKE5_INPUT_JSONL)
smoke_gold_rows = read_jsonl(SMOKE5_GOLD_JSONL)
assert len(input_rows) == 1 and len(gold_rows) == 1
assert len(smoke_input_rows) == 5 and len(smoke_gold_rows) == 5
assert "entities" not in input_rows[0] and "relations" not in input_rows[0]
assert all("entities" not in row and "relations" not in row for row in smoke_input_rows)
assert input_rows[0]["document_id"] == gold_rows[0]["document_id"]
assert [x["document_id"] for x in smoke_input_rows] == [x["document_id"] for x in smoke_gold_rows]

profile = read_json(PROFILE_PATH)
task = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)
gold = gold_rows[0]
seed_summary = seed_ontology_summary(ONTOLOGY_PATH)
l1_cfg = profile["layers"]["layer01_linguistic_expression_extraction"]
l2_cfg = profile["layers"]["layer02_candidate_enrichment"]

print("Document:", input_rows[0]["document_id"], "-", input_rows[0]["title"])
print("Source characters:", len(input_rows[0]["text"]))
print("Sentences:", len(input_rows[0]["sentences"]))
print("Source tokens:", sum(len(x) for x in input_rows[0]["tokens"]))
print("Gold events (not exposed to pipeline):", len(gold["entities"]))
print("Gold evaluated relations:", sum(len(v) for k, v in gold["relations"].items() if k in RELATION_IDS))
print("Ignored null pairs:", len(gold["relations"].get("null", [])))
print("OWL-Time classes loaded:", seed_summary["class_count"])
print("OWL-Time properties loaded:", seed_summary["property_count"])
print("Controlled task relations:", catalog["property_count"])
print("Relation IDs:", task["allowed_relation_ids"])
print("Layer 1 sentence workers:", l1_cfg["sentence_workers"])
print("Layer 1 coverage reviews:", l1_cfg["coverage_review_passes"])
print("Pair strategy:", l1_cfg["pair_strategy"])
print("Exhaustive-event threshold:", l1_cfg["exhaustive_event_threshold"])
print("Relation-generation LLM calls:", l1_cfg["relation_generation_llm_calls"])
print("Layer 2 pair batch size:", l2_cfg["pair_batch_size"])
print("Layer 2 pair batch workers:", l2_cfg["pair_batch_workers"])
print("Five-way decisions:", l2_cfg["five_way_decisions"])
print("Positive evidence required:", l2_cfg["evidence_required_for_positive"])
print("NONE filtered before Layer 3:", l2_cfg["filter_none_before_layer03"])
print("Global-trigger projection fallback:", profile["benchmark_projection"]["fallback_unique_trigger"])
print("Mention-free relation schemas injected:", len(profile["relations"]["allowed"]))
print("Five-document input IDs:", [row["document_id"] for row in smoke_input_rows])

assert seed_summary["class_count"] > 0 or seed_summary["property_count"] > 0
assert catalog["property_count"] == 2
assert set(task["allowed_relation_ids"]) == set(RELATION_IDS)
assert profile["relations"]["allowed"] == []
assert l1_cfg["relation_generation_llm_calls"] == 0
assert l1_cfg["deterministic_unordered_pair_pool"] is True
assert l2_cfg["filter_none_before_layer03"] is True
assert l2_cfg["evidence_required_for_positive"] is True
assert profile["benchmark_projection"]["fallback_unique_trigger"] is False
assert profile["anti_cheating"]["direct_eventstoryline_extraction"] is False
assert profile["anti_cheating"]["source_event_anchoring"] is False
assert profile["anti_cheating"]["gold_pair_hints"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False
assert profile["benchmark_projection"]["same_seed_ontology_as_ragtree"] is True

# Prompt examples are synthetic and schema-exact; no dataset gold pair is embedded.
assert len(task["five_way_schema_examples"]) >= 4
assert {x["output"]["decision"] for x in task["five_way_schema_examples"]} >= {
    "A_PRECONDITION_B", "A_FALLING_ACTION_B", "B_PRECONDITION_A", "NONE"
}

display(Markdown("### Indexed source table used by Layer 1"))
print(indexed_token_table(input_rows[0]["sentences"], input_rows[0]["tokens"]))

assert profile["anti_cheating"]["gold_event_lexicon"] is False
assert profile["anti_cheating"]["gold_event_count"] is False
assert profile["anti_cheating"]["gold_relation_count"] is False

print("Atomic span review enabled:", l1_cfg.get("atomic_review_enabled"))
print("Pair-local direct-link mode:", l2_cfg.get("direct_link_only"))
print("Candidate closure enabled:", l2_cfg.get("closure_enabled"))
print("Targeted NONE review:", l2_cfg.get("none_review_enabled"), "max pairs =", l2_cfg.get("none_review_max_pairs"))
print("Positive verifier:", l2_cfg.get("positive_verifier_enabled"))


Document: EventStoryLine - 1_10ecbplus - 1_10ecbplus
Source characters: 745
Sentences: 6
Source tokens: 147
Gold events (not exposed to pipeline): 15
Gold evaluated relations: 20
Ignored null pairs: 1
OWL-Time classes loaded: 23
OWL-Time properties loaded: 62
Controlled task relations: 2
Relation IDs: ['PRECONDITION', 'FALLING_ACTION']
Layer 1 sentence workers: 8
Layer 1 coverage reviews: 3
Pair strategy: adaptive
Exhaustive-event threshold: 25
Relation-generation LLM calls: 0
Layer 2 pair batch size: 8
Layer 2 pair batch workers: 8
Five-way decisions: ['A_PRECONDITION_B', 'A_FALLING_ACTION_B', 'B_PRECONDITION_A', 'B_FALLING_ACTION_A', 'NONE']
Positive evidence required: True
NONE filtered before Layer 3: True
Global-trigger projection fallback: False
Mention-free relation schemas injected: 0
Five-document input IDs: ['EventStoryLine - 1_10ecbplus', 'EventStoryLine - 1_11ecbplus', 'EventStoryLine - 1_12ecbplus', 'EventStoryLine - 1_13ecbplus', 'EventStoryLine - 1_14ecbplus']


### Indexed source table used by Layer 1

[S0] http : / / articles . latimes . com / 2013 / may / 03 / local / la - me - 0504 - lohan - rehab - 20130504
TOKENS 0=http 1=: 2=/ 3=/ 4=articles 5=. 6=latimes 7=. 8=com 9=/ 10=2013 11=/ 12=may 13=/ 14=03 15=/ 16=local 17=/ 18=la 19=- 20=me 21=- 22=0504 23=- 24=lohan 25=- 26=rehab 27=- 28=20130504

[S1] Lindsay Lohan checks into Betty Ford Center
TOKENS 0=Lindsay 1=Lohan 2=checks 3=into 4=Betty 5=Ford 6=Center

[S2] May 03 , 2013
TOKENS 0=May 1=03 2=, 3=2013

[S3] After skipping out on entering a Newport Beach rehabilitation facility and facing the prospect of arrest for violating her probation , Lindsay Lohan has checked into the Betty Ford Center to begin a 90 - day court - mandated stay in her reckless driving conviction .
TOKENS 0=After 1=skipping 2=out 3=on 4=entering 5=a 6=Newport 7=Beach 8=rehabilitation 9=facility 10=and 11=facing 12=the 13=prospect 14=of 15=arrest 16=for 17=violating 18=her 19=probation 20=, 21=Lindsay 22=Lohan 23=has 24=checked 25=into 26=the 27=Betty 28=Fo

## 3. Run the full native Layer 0--12 pipeline

In [8]:
if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    final_state = run_native_pipeline(
        project_root=PROJECT_ROOT,
        input_jsonl=INPUT_JSONL,
        ontology_path=ONTOLOGY_PATH,
        profile_path=PROFILE_PATH,
        guidance_path=GUIDANCE_PATH,
        task_guidance_path=TASK_GUIDANCE_PATH,
        relation_catalog_path=RELATION_CATALOG,
        relation_aliases_path=RELATION_ALIASES,
        run_dir=RUN_DIR,
        model_name=MODEL_NAME,
        api_key=API_KEY,
        host=OPENROUTER_HOST,
        workers=WORKERS,
        reasoning_effort=REASONING_EFFORT,
        verbose=True,
        clean_run_dir=CLEAN_RUN_DIR,
    )
    print("Full native run completed.")
else:
    print("RUN_PIPELINE=False: reusing", RUN_DIR)

[NeoOLAF] Run directory: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v1_5_owltime_atomic_verified
[NeoOLAF] from_layer=0, to_layer=None, skip_layers=None
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer00_preprocessing', 'layer01_linguistic_expression_extraction', 'layer02_candidate_enrichment', 'layer03_candidate_typing_resolution', 'layer04_candidate_relation_extraction', 'layer05_candidate_triple_generation', 'layer06_concept_relation_induction', 'layer07_hierarchisation', 'layer08_axiom_schemata_extraction', 'layer09_general_axiom_extraction', 'layer10_validation_reasoning', 'layer11_inference_completion', 'layer12_serialization']
[NeoOLAF] Layer 0/12: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.00s
[NeoOLAF] Layer 1/12: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_ling

[NeoOLAF] Finished layer: layer03_candidate_typing_resolution in 0.01s
[NeoOLAF] Layer 4/12: layer04_candidate_relation_extraction

[NeoOLAF] Starting layer: layer04_candidate_relation_extraction
[NeoOLAF][Layer 4] strategy=structured_exact_then_native_parallel_fallback; parallel_workers=8; attempts=1
[NeoOLAF] Finished layer: layer04_candidate_relation_extraction in 0.01s
[NeoOLAF] Layer 5/12: layer05_candidate_triple_generation

[NeoOLAF] Starting layer: layer05_candidate_triple_generation


[NeoOLAF] Finished layer: layer05_candidate_triple_generation in 0.00s
[NeoOLAF] Layer 6/12: layer06_concept_relation_induction

[NeoOLAF] Starting layer: layer06_concept_relation_induction
[NeoOLAF][Layer 6] deterministic ontology-aware concept induction for 19 node candidates; no LLM calls.
[NeoOLAF][Layer 6] deterministic ontology-aware relation induction for 13 relation candidates; no LLM calls.
[NeoOLAF] Finished layer: layer06_concept_relation_induction in 0.00s
[NeoOLAF] Layer 7/12: layer07_hierarchisation

[NeoOLAF] Starting layer: layer07_hierarchisation
[NeoOLAF] Finished layer: layer07_hierarchisation in 0.00s
[NeoOLAF] Layer 8/12: layer08_axiom_schemata_extraction

[NeoOLAF] Starting layer: layer08_axiom_schemata_extraction
[NeoOLAF][Layer 8] strategy=ontology_aware_axiom_schema_generation
[NeoOLAF] Finished layer: layer08_axiom_schemata_extraction in 0.00s
[NeoOLAF] Layer 9/12: layer09_general_axiom_extraction

[NeoOLAF] Starting layer: layer09_general_axiom_extraction
[Ne

[NeoOLAF] Finished layer: layer10_validation_reasoning in 0.00s
[NeoOLAF] Layer 11/12: layer11_inference_completion

[NeoOLAF] Starting layer: layer11_inference_completion
[NeoOLAF][Layer 11] strategy=ontology_aware_semantic_completion
[NeoOLAF][Layer 11] deterministic completion; max_concurrency=16; no LLM calls.
[NeoOLAF] Finished layer: layer11_inference_completion in 0.00s
[NeoOLAF] Layer 12/12: layer12_serialization

[NeoOLAF] Starting layer: layer12_serialization
[NeoOLAF] Exports written to: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v1_5_owltime_atomic_verified/exports
[NeoOLAF] Finished layer: layer12_serialization in 0.04s
[NeoOLAF] Pipeline finished in 541.21s
[NeoOLAF] Saved checkpoint: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/eventstoryline_native_layer_ablation/document_1_10ecbplus_v1_5_owltime_atomic_verified/checkpoints/after_selected_pipeline.pkl.gz
[Ne

### Runtime evidence saved

- `run_manifest.json` records OWL-Time and the v1.5 operational/scientific fingerprint.
- `run_logs/layer01_event_inventory.json` records all accepted/rejected event spans.
- `run_logs/layer01_pair_pool.json` records the deterministic initial unordered pair pool.
- `run_logs/layer02_closure_pair_pool.json` records optional two-hop candidate-closure pairs.
- `run_logs/layer02_relation_decisions.json` records accepted, `NONE`, invalid, reversed and recovered decisions.
- `run_logs/layer02_compact_prompt_audit.json` and `layer02_batch_audit.json` record bounded batch execution.
- `run_logs/layer02_batch_cache/` stores resumable prompt-fingerprint responses.
- Layer 3 receives only validated positive controlled relations; `NONE` never propagates.
- Layer 2/4 decisions, ontology retrieval, API responses, and all Layer 0--12 states remain saved.


## 4. Projected benchmark metrics and strict native span metrics


In [9]:
summary = analyze_run(
    run_dir=RUN_DIR,
    gold_jsonl=GOLD_JSONL,
    catalog_path=RELATION_CATALOG,
    aliases_path=RELATION_ALIASES,
)

display(pd.DataFrame(summary["layer_summary"]))

print("Projected benchmark relation evaluation; exact-span projection, null excluded")
display(pd.DataFrame([summary["strict_relation_evaluation"]]))

print("Strict native span relation evaluation; unmapped native predictions count as false positives")
display(pd.DataFrame([summary["native_span_relation_evaluation"]]))

print("Projected event-ID evaluation")
display(pd.DataFrame([summary["event_entity_evaluation"]]))

print("Strict native event-span evaluation")
display(pd.DataFrame([summary["native_span_event_evaluation"]]))

print("Relation-endpoint projected event evaluation")
display(pd.DataFrame([summary["relation_endpoint_evaluation"]]))

print("Relation candidate-pool coverage")
display(pd.DataFrame([summary["candidate_pool_evaluation"]]))

print("Per-relation projected metrics")
display(pd.DataFrame(summary["per_relation_metrics"]))

print("Direction/class confusion matrix")
display(pd.DataFrame(summary["relation_confusion_matrix"]))

print("Cumulative evaluation")
display(pd.DataFrame(summary["cumulative_evaluation"]))

print("First-failure counts")
print(summary["failure_counts"])


,layer,layer_name,linguistic_expressions,enriched_expressions,entity_candidates,relation_candidates,attribute_candidates,event_candidates,candidate_relation_assertions,candidate_triples,concept_candidates,ontology_relation_candidates,concept_hierarchy_links,relation_hierarchy_links,axiom_schema_candidates,general_axiom_candidates,completion_candidates,validation_issues,reasoning_inferred_triples
0,0,layer00_preprocessing,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,layer01_linguistic_expression_extraction,190,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,layer02_candidate_enrichment,190,32,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,layer03_candidate_typing_resolution,190,32,0,13,0,19,0,0,0,0,0,0,0,0,0,0,0
4,4,layer04_candidate_relation_extraction,190,32,0,13,0,19,13,0,0,0,0,0,0,0,0,0,0
5,5,layer05_candidate_triple_generation,190,32,0,13,0,19,13,13,0,0,0,0,0,0,0,0,0
6,6,layer06_concept_relation_induction,190,32,0,13,0,19,13,13,0,2,0,0,0,0,0,0,0
7,7,layer07_hierarchisation,190,32,0,13,0,19,13,13,0,2,0,2,0,0,0,0,0
8,8,layer08_axiom_schemata_extraction,190,32,0,13,0,19,13,13,0,2,0,2,6,0,0,0,0
9,9,layer09_general_axiom_extraction,190,32,0,13,0,19,13,13,0,2,0,2,6,8,0,0,0


Projected benchmark relation evaluation; exact-span projection, null excluded


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,7,20,0,7,20,0.0,0.0,0.0


Strict native span relation evaluation; unmapped native predictions count as false positives


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,13,20,0,13,20,0.0,0.0,0.0


...

Direction/class confusion matrix


,gold_relation,PRECONDITION,FALLING_ACTION,NONE,REVERSED_PRECONDITION,REVERSED_FALLING_ACTION,PAIR_MISSING,INVALID
0,PRECONDITION,0,1,0,0,0,3,3
1,FALLING_ACTION,1,0,8,1,0,3,0


Cumulative evaluation


,layer,layer_name,projected_relation_predicted,projected_relation_gold,projected_relation_true_positive,projected_relation_false_positive,projected_relation_false_negative,projected_relation_precision,projected_relation_recall,projected_relation_f1,...,projected_event_recall,projected_event_f1,native_span_event_predicted,native_span_event_gold,native_span_event_true_positive,native_span_event_false_positive,native_span_event_false_negative,native_span_event_precision,native_span_event_recall,native_span_event_f1
0,0,layer00_preprocessing,0,20,0,0,20,0.0,0.0,0.0,...,0.000000,0.000000,0,15,0,0,15,0.000000,0.000000,0.000000
1,1,layer01_linguistic_expression_extraction,0,20,0,0,20,0.0,0.0,0.0,...,0.866667,0.928571,19,15,13,6,2,0.684211,0.866667,0.764706
2,2,layer02_candidate_enrichment,0,20,0,0,20,0.0,0.0,0.0,...,0.866667,0.928571,19,15,13,6,2,0.684211,0.866667,0.764706
3,3,layer03_candidate_typing_resolution,0,20,0,0,20,0.0,0.0,0.0,...,0.866667,0.928571,19,15,13,6,2,0.684211,0.866667,0.764706
4,4,layer04_candidate_relation_extraction,7,20,0,7,20,0.0,0.0,0.0,...,0.866667,0.928571,19,15,13,6,2,0.684211,0.866667,0.764706
5,5,layer05_candidate_triple_generation,7,20,0,7,20,0.0,0.0,0.0,...,0.866667,0.928571,19,15,13,6,2,0.684211,0.866667,0.764706
6,6,layer06_concept_relation_induction,7,20,0,7,20,0.0,0.0,0.0,...,0.866667,0.928571,19,15,13,6,2,0.684211,0.866667,0.764706
7,7,layer07_hierarchisation,7,20,0,7,20,0.0,0.0,0.0,...,0.866667,0.928571,19,15,13,6,2,0.684211,0.866667,0.764706
8,8,layer08_axiom_schemata_extraction,7,20,0,7,20,0.0,0.0,0.0,...,0.866667,0.928571,19,15,13,6,2,0.684211,0.866667,0.764706
9,9,layer09_general_axiom_extraction,7,20,0,7,20,0.0,0.0,0.0,...,0.866667,0.928571,19,15,13,6,2,0.684211,0.866667,0.764706


First-failure counts
{'classified_none': 8, 'filtered_invalid_output': 3, 'target_event_missing': 3, 'wrong_direction': 1, 'wrong_relation_class': 2, 'source_event_missing': 3}


## 5. Layer 1 validated event inventory and deterministic unordered relation-pair pool


In [10]:
layer1_rows = read_json(RUN_DIR / "run_logs/layer01_event_relation_instances.json")
layer1_df = pd.DataFrame(layer1_rows)
display(layer1_df)

accepted = layer1_df[layer1_df["status"] == "accepted"] if not layer1_df.empty else layer1_df
if not accepted.empty:
    print("Accepted event mentions:", int((accepted["label"] == "event_mention").sum()))
    print("Deterministic pair expressions:", int((accepted["label"] == "relation_instance").sum()))
    print("Rejected rows:", int((layer1_df["status"] == "rejected").sum()))


,phase,status,expr_id,text,label,pair_id,candidate_reasons
0,atomic_materialization,accepted,expr_00000,S1[2:4]::checks into,event_mention,NaN,NaN
1,atomic_materialization,accepted,expr_00001,S3[1:3]::skipping out,event_mention,NaN,NaN
2,atomic_materialization,accepted,expr_00002,S3[4:5]::entering,event_mention,NaN,NaN
3,atomic_materialization,accepted,expr_00003,S3[11:12]::facing,event_mention,NaN,NaN
4,atomic_materialization,accepted,expr_00004,S3[15:16]::arrest,event_mention,NaN,NaN
...,...,...,...,...,...,...,...
185,atomic_pair_pool_materialization,accepted,expr_00185,S5[9:12]::rear - ended || potentially related ...,relation_instance,P00166,[exhaustive_unordered_pair]
186,atomic_pair_pool_materialization,accepted,expr_00186,S5[9:12]::rear - ended || potentially related ...,relation_instance,P00167,[exhaustive_unordered_pair]
187,atomic_pair_pool_materialization,accepted,expr_00187,S5[27:28]::lied || potentially related || S5[3...,relation_instance,P00168,[exhaustive_unordered_pair]
188,atomic_pair_pool_materialization,accepted,expr_00188,S5[27:28]::lied || potentially related || S5[3...,relation_instance,P00169,[exhaustive_unordered_pair]


Accepted event mentions: 19
Deterministic pair expressions: 171
Rejected rows: 0


### Layer 1A span validation/repair and Layer 1 deterministic pair audit


In [11]:
inventory_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_event_inventory.json"))
atomic_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_atomic_span_review.json"))
pair_pool = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_pair_pool.json"))
call_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_call_audit.json"))

print("Final atomic event inventory audit")
display(inventory_audit)
print("Atomic KEEP/DROP/REPLACE/SPLIT audit")
display(atomic_audit)

print("Deterministic unordered pair pool rebuilt after atomic refinement")
display(pair_pool)
if not pair_pool.empty:
    print("Initial pair candidates:", len(pair_pool))
    print("Pair strategy values:", sorted(pair_pool["strategy"].dropna().unique()))

print("Layer 1 sentence/review/atomic/pair construction audit")
display(call_audit)
if not call_audit.empty:
    display(call_audit.groupby(["phase", "status"], dropna=False).size().reset_index(name="calls"))


Final atomic event inventory audit


,phase,original_event_key,result_event_key,action,reasons,event_key,reason,status
0,deterministic_atomic_minimization,S1[2:4]::checks into,S1[2:4]::checks into,KEEP,[],NaN,NaN,NaN
1,deterministic_atomic_minimization,S3[1:3]::skipping out,S3[1:3]::skipping out,KEEP,[],NaN,NaN,NaN
2,deterministic_atomic_minimization,S3[4:5]::entering,S3[4:5]::entering,KEEP,[],NaN,NaN,NaN
3,deterministic_atomic_minimization,S3[11:12]::facing,S3[11:12]::facing,KEEP,[],NaN,NaN,NaN
4,deterministic_atomic_minimization,S3[15:16]::arrest,S3[15:16]::arrest,KEEP,[],NaN,NaN,NaN
5,deterministic_atomic_minimization,S3[17:18]::violating,S3[17:18]::violating,KEEP,[],NaN,NaN,NaN
6,deterministic_atomic_minimization,S3[24:26]::checked into,S3[24:26]::checked into,KEEP,[],NaN,NaN,NaN
7,deterministic_atomic_minimization,S3[31:32]::begin,S3[31:32]::begin,KEEP,[],NaN,NaN,NaN
8,deterministic_atomic_minimization,S3[39:40]::stay,S3[39:40]::stay,KEEP,[],NaN,NaN,NaN
9,deterministic_atomic_minimization,S3[43:44]::driving,S3[43:44]::driving,KEEP,[],NaN,NaN,NaN


Atomic KEEP/DROP/REPLACE/SPLIT audit


,phase,original_event_key,result_event_key,action,reasons,event_key,reason,status
0,deterministic_atomic_minimization,S1[2:4]::checks into,S1[2:4]::checks into,KEEP,[],NaN,NaN,NaN
1,deterministic_atomic_minimization,S3[1:3]::skipping out,S3[1:3]::skipping out,KEEP,[],NaN,NaN,NaN
2,deterministic_atomic_minimization,S3[4:5]::entering,S3[4:5]::entering,KEEP,[],NaN,NaN,NaN
3,deterministic_atomic_minimization,S3[11:12]::facing,S3[11:12]::facing,KEEP,[],NaN,NaN,NaN
4,deterministic_atomic_minimization,S3[15:16]::arrest,S3[15:16]::arrest,KEEP,[],NaN,NaN,NaN
5,deterministic_atomic_minimization,S3[17:18]::violating,S3[17:18]::violating,KEEP,[],NaN,NaN,NaN
6,deterministic_atomic_minimization,S3[24:26]::checked into,S3[24:26]::checked into,KEEP,[],NaN,NaN,NaN
7,deterministic_atomic_minimization,S3[31:32]::begin,S3[31:32]::begin,KEEP,[],NaN,NaN,NaN
8,deterministic_atomic_minimization,S3[39:40]::stay,S3[39:40]::stay,KEEP,[],NaN,NaN,NaN
9,deterministic_atomic_minimization,S3[43:44]::driving,S3[43:44]::driving,KEEP,[],NaN,NaN,NaN


Deterministic unordered pair pool rebuilt after atomic refinement


,event_a_id,event_b_id,event_a_key,event_b_key,event_a_sentence,event_b_sentence,candidate_reasons,strategy,phase,pair_id
0,E0000,E0001,S1[2:4]::checks into,S3[1:3]::skipping out,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00000
1,E0000,E0002,S1[2:4]::checks into,S3[4:5]::entering,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00001
2,E0000,E0003,S1[2:4]::checks into,S3[11:12]::facing,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00002
3,E0000,E0004,S1[2:4]::checks into,S3[15:16]::arrest,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00003
4,E0000,E0005,S1[2:4]::checks into,S3[17:18]::violating,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00004
...,...,...,...,...,...,...,...,...,...,...
166,E0015,E0017,S5[9:12]::rear - ended,S5[31:32]::telling,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00166
167,E0015,E0018,S5[9:12]::rear - ended,S5[36:37]::driving,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00167
168,E0016,E0017,S5[27:28]::lied,S5[31:32]::telling,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00168
169,E0016,E0018,S5[27:28]::lied,S5[36:37]::driving,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00169


Initial pair candidates: 171
Pair strategy values: ['exhaustive']
Layer 1 sentence/review/atomic/pair construction audit


,phase,sentence_id,status,reason,proposals,new_validated_events,pass_index,inventory_size_after,strategy,event_count,pair_count,exhaustive_event_threshold,input_events,reviewed_events,dropped_events,replacement_or_added_events,final_events
0,sentence_inventory,0.0,skipped,url_sentence,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sentence_inventory,2.0,skipped,date_or_publication_metadata,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sentence_inventory,1.0,ok,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sentence_inventory,3.0,ok,NaN,10.0,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sentence_inventory,4.0,ok,NaN,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,sentence_inventory,5.0,ok,NaN,5.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,coverage_review,NaN,ok,NaN,1.0,0.0,1.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,coverage_review,NaN,ok,NaN,1.0,0.0,2.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,coverage_review,NaN,ok,NaN,2.0,1.0,3.0,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,deterministic_pair_pool,NaN,ok,NaN,NaN,NaN,NaN,NaN,exhaustive,19.0,171.0,25.0,NaN,NaN,NaN,NaN,NaN


,phase,status,calls
0,atomic_span_review,ok,1
1,coverage_review,ok,3
2,deterministic_pair_pool,ok,1
3,deterministic_pair_pool_after_atomic_review,ok,1
4,sentence_inventory,ok,4
5,sentence_inventory,skipped,2


## 6. Batched five-way relation decisions, recovery, closure and filtering


In [12]:
layer2 = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_relation_decisions.json"))
prompt_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_compact_prompt_audit.json"))
batch_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_batch_audit.json"))
none_review = read_json(RUN_DIR / "run_logs/layer02_none_review.json")
positive_verification = read_json(RUN_DIR / "run_logs/layer02_positive_verification.json")
closure_pool = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_closure_pair_pool.json"))

display(layer2)
if not layer2.empty:
    print("Final Layer 2 decision status counts")
    display(layer2.groupby(["status", "decision"], dropna=False).size().reset_index(name="pairs"))
    accepted_l2 = layer2[layer2["status"] == "accepted"]
    print("Verified positive relations entering Layer 3:", len(accepted_l2))
    print("Filtered NONE/invalid relations:", len(layer2) - len(accepted_l2))

print("Pair-local primary prompt audit")
display(prompt_audit)
if not prompt_audit.empty:
    print("Prompted pair decisions:", int(prompt_audit["pair_count"].sum()))
    print("Mean pairs per prompt:", round(prompt_audit["pair_count"].mean(), 2))
    print("Mean Layer 2 user characters:", round(prompt_audit["user_chars"].mean(), 1))
    print("Maximum Layer 2 user characters:", int(prompt_audit["user_chars"].max()))

print("Primary batch/recovery/cache audit")
display(batch_audit)
print("Targeted false-NONE review audit")
display(pd.DataFrame(none_review.get("batches", [])))
print("Selected false-NONE review pairs:", len(none_review.get("selected_pairs", [])))
print("Positive verification audit")
display(pd.DataFrame(positive_verification.get("batches", [])))
print("Input positives sent to verifier:", len(positive_verification.get("input_positive_pairs", [])))
print("Closure pool (must be empty in v1.5 direct-link mode)")
display(closure_pool)


,pair_id,phase,event_a,event_b,decision,found,status,evidence_sentence_ids,evidence_text,reason,...,recovery,raw_decisions,normalized_decisions,expr_id,source,target,selected_relation_id,ontology_hints,evidence_grounded,allowed_context_sentence_ids
0,P00000,initial,S1[2:4]::checks into,S3[1:3]::skipping out,NONE,False,filtered_none,[],,Skipping out occurs before checking into and i...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,P00000,none_review,NaN,NaN,NaN,NaN,missing_decision,NaN,NaN,NaN,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,P00001,initial,S1[2:4]::checks into,S3[4:5]::entering,NONE,False,filtered_none,[],,Entering a rehabilitation facility is part of ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,P00001,none_review,NaN,NaN,NaN,NaN,missing_decision,NaN,NaN,NaN,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,P00002,initial,S1[2:4]::checks into,S3[11:12]::facing,NONE,False,filtered_none,[],,Facing the prospect of arrest occurs before ch...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215,P00168,initial,S5[27:28]::lied,S5[31:32]::telling,B_PRECONDITION_A,True,accepted,[5],"and then lied to police, telling them she was ...",Positive verifier: The telling is a component ...,...,NaN,NaN,NaN,expr_00187,S5[31:32]::telling,S5[27:28]::lied,PRECONDITION,"[controlled_relation:PRECONDITION, promote_to_...",NaN,NaN
216,P00169,initial,S5[27:28]::lied,S5[36:37]::driving,NONE,False,filtered_none,[],,The sentence states that she lied about not dr...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
217,P00169,none_review,NaN,NaN,NaN,NaN,missing_decision,NaN,NaN,NaN,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
218,P00170,initial,S5[31:32]::telling,S5[36:37]::driving,NONE,False,filtered_none,[],,The telling merely reports that she was not dr...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Final Layer 2 decision status counts


,status,decision,pairs
0,accepted,A_FALLING_ACTION_B,3
1,accepted,A_PRECONDITION_B,4
2,accepted,B_PRECONDITION_A,6
3,conflicting_or_invalid_decisions,NaN,16
4,filtered_invalid_output,NONE,16
5,filtered_none,NONE,142
6,invalid_missing_or_ungrounded_evidence,A_PRECONDITION_B,1
7,missing_decision,NaN,32


Verified positive relations entering Layer 3: 13
Filtered NONE/invalid relations: 207
Pair-local primary prompt audit


,batch_index,phase,recovery,pair_ids,pair_count,prompt_kind,candidate_decisions,system_chars,user_chars
0,0,primary_direct,False,"[P00000, P00001, P00002, P00003, P00004, P0000...",8,pair_local_direct_primary,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",1910,30783
1,1,primary_direct,False,"[P00008, P00009, P00010, P00011, P00012, P0001...",8,pair_local_direct_primary,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",1910,36522
2,2,primary_direct,False,"[P00016, P00017, P00018, P00019, P00020, P0002...",8,pair_local_direct_primary,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",1910,26955
3,3,primary_direct,False,"[P00024, P00025, P00026, P00027, P00028, P0002...",8,pair_local_direct_primary,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",1910,27909
4,4,primary_direct,False,"[P00032, P00033, P00034, P00035, P00036, P0003...",8,pair_local_direct_primary,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",1910,25976
5,5,primary_direct,False,"[P00040, P00041, P00042, P00043, P00044, P0004...",8,pair_local_direct_primary,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",1910,27877
6,6,primary_direct,False,"[P00048, P00049, P00050, P00051, P00052, P0005...",8,pair_local_direct_primary,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",1910,25962
7,7,primary_direct,False,"[P00056, P00057, P00058, P00059, P00060, P0006...",8,pair_local_direct_primary,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",1910,28834
8,8,primary_direct,False,"[P00064, P00065, P00066, P00067, P00068, P0006...",8,pair_local_direct_primary,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",1910,25012
9,9,primary_direct,False,"[P00072, P00073, P00074, P00075, P00076, P0007...",8,pair_local_direct_primary,"[A_PRECONDITION_B, A_FALLING_ACTION_B, B_PRECO...",1910,30745


...

,batch_index,pair_count,positive_revisions,unresolved,status,attempt,cache_path
0,1,8,0,[],ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
1,2,8,0,[],ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
2,4,8,0,"[P00107, P00116, P00117, P00126, P00143, P0016...",ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
3,3,8,0,[],ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
4,6,8,0,[],ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
5,5,8,0,"[P00167, P00169, P00170, P00010, P00011, P0001...",ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
6,0,8,0,"[P00128, P00155, P00000, P00001, P00002, P0000...",ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
7,7,8,0,"[P00045, P00058, P00059, P00060, P00072, P0007...",ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...


Selected false-NONE review pairs: 64
Positive verification audit


,batch_index,pair_count,kept_positive,unresolved,status,attempt,cache_path
0,1,8,0,"[P00038, P00080, P00081, P00082, P00088, P0009...",ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
1,0,8,0,"[P00024, P00025, P00026, P00005, P00006, P0000...",ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
2,2,8,8,[],ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
3,3,6,5,[],ok,0,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...


Input positives sent to verifier: 30
Closure pool (must be empty in v1.5 direct-link mode)


""


## 7. Exact gold-relation failure trace and confusion matrix


In [13]:
trace_df = pd.read_csv(RUN_DIR / "analysis/gold_relation_trace.csv")
confusion_df = pd.read_csv(RUN_DIR / "analysis/relation_confusion_matrix.csv")
display(trace_df)
display(
    trace_df.groupby("first_failure", dropna=False)
    .size()
    .reset_index(name="gold_relations")
    .sort_values("gold_relations", ascending=False)
)
display(confusion_df)


,source_key,relation_id,target_key,first_failure,pair_in_pool,decision_status,predicted_decision,predicted_source,predicted_relation,predicted_target
0,S1[2:4]::checks into,FALLING_ACTION,S3[15:16]::arrest,classified_none,True,filtered_none,NONE,NaN,NaN,NaN
1,S1[2:4]::checks into,FALLING_ACTION,S3[4:5]::entering,classified_none,True,filtered_none,NONE,NaN,NaN,NaN
2,S1[2:4]::checks into,PRECONDITION,S3[39:40]::stay,filtered_invalid_output,True,filtered_invalid_output,NONE,NaN,NaN,NaN
3,S1[2:4]::checks into,PRECONDITION,S5[2:3]::stay,target_event_missing,False,NaN,NaN,NaN,NaN,NaN
4,S3[15:16]::arrest,FALLING_ACTION,S3[17:18]::violating,wrong_direction,True,accepted,B_PRECONDITION_A,S3[17:18]::violating,PRECONDITION,S3[15:16]::arrest
5,S3[15:16]::arrest,PRECONDITION,S3[24:26]::checked into,wrong_relation_class,True,accepted,A_FALLING_ACTION_B,S3[15:16]::arrest,FALLING_ACTION,S3[24:26]::checked into
6,S3[24:26]::checked into,PRECONDITION,S3[39:40]::stay,filtered_invalid_output,True,filtered_invalid_output,NONE,NaN,NaN,NaN
7,S3[24:26]::checked into,PRECONDITION,S5[2:3]::stay,target_event_missing,False,NaN,NaN,NaN,NaN,NaN
8,S3[39:40]::stay,FALLING_ACTION,S3[44:45]::conviction,classified_none,True,filtered_none,NONE,NaN,NaN,NaN
9,S3[39:40]::stay,FALLING_ACTION,S5[27:28]::lied,classified_none,True,filtered_none,NONE,NaN,NaN,NaN


,first_failure,gold_relations
0,classified_none,8
1,filtered_invalid_output,3
2,source_event_missing,3
3,target_event_missing,3
5,wrong_relation_class,2
4,wrong_direction,1


,gold_relation,PRECONDITION,FALLING_ACTION,NONE,REVERSED_PRECONDITION,REVERSED_FALLING_ACTION,PAIR_MISSING,INVALID
0,PRECONDITION,0,1,0,0,0,3,3
1,FALLING_ACTION,1,0,8,1,0,3,0


## 8. Native event candidates, assertions and triples

In [14]:
states = {index: state for index, _, state in load_layer_states(RUN_DIR)}
layer3 = states.get(3)
layer4 = states.get(4)
layer5 = states.get(5)
layer6 = states.get(6)
layer11 = states.get(11)

if layer3:
    print("Layer 3 event candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "ontology_hints": c.ontology_hints,
    } for c in layer3.event_candidates or []]))

    print("Layer 3 relation candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "controlled_hints": [h for h in c.ontology_hints if str(h).lower().startswith("controlled_relation:")],
    } for c in layer3.relation_candidates or []]))
    assert all(c.mentions for c in layer3.relation_candidates or []), "Mention-free relation candidate detected."

if layer4:
    print("Layer 4 assertions")
    display(pd.DataFrame([{
        "source": x.source_candidate_label,
        "predicate": x.relation_label,
        "target": x.target_candidate_label,
        "confidence": x.confidence,
    } for x in layer4.candidate_relation_assertions or []]))

if layer5:
    print("Layer 5 triples")
    display(pd.DataFrame([{
        "subject": x.subject_label,
        "predicate": x.predicate_label,
        "object": x.object_label,
        "confidence": x.confidence,
    } for x in layer5.candidate_triples or []]))

print("Layer 6 ontology relation candidates:", len(layer6.ontology_relation_candidates or []) if layer6 else None)
print("Layer 11 completion candidates:", len(layer11.completion_candidates or []) if layer11 else None)

Layer 3 event candidates


,candidate_id,canonical_label,mentions,ontology_hints
0,cand_s_00000,S1[2:4]::checks into,[S1[2:4]::checks into],"[semantic_role:event_mention, candidate_family..."
1,cand_s_00001,S3[1:3]::skipping out,[S3[1:3]::skipping out],"[semantic_role:event_mention, candidate_family..."
2,cand_s_00002,S3[4:5]::entering,[S3[4:5]::entering],"[semantic_role:event_mention, candidate_family..."
3,cand_s_00003,S3[11:12]::facing,[S3[11:12]::facing],"[semantic_role:event_mention, candidate_family..."
4,cand_s_00004,S3[15:16]::arrest,[S3[15:16]::arrest],"[semantic_role:event_mention, candidate_family..."
5,cand_s_00005,S3[17:18]::violating,[S3[17:18]::violating],"[semantic_role:event_mention, candidate_family..."
6,cand_s_00006,S3[24:26]::checked into,[S3[24:26]::checked into],"[semantic_role:event_mention, candidate_family..."
7,cand_s_00007,S3[31:32]::begin,[S3[31:32]::begin],"[semantic_role:event_mention, candidate_family..."
8,cand_s_00008,S3[39:40]::stay,[S3[39:40]::stay],"[semantic_role:event_mention, candidate_family..."
9,cand_s_00009,S3[43:44]::driving,[S3[43:44]::driving],"[semantic_role:event_mention, candidate_family..."


Layer 3 relation candidates


,candidate_id,canonical_label,mentions,controlled_hints
0,cand_r_00000,PRECONDITION,[S5[36:37]::driving || PRECONDITION || S3[11:1...,[controlled_relation:PRECONDITION]
1,cand_r_00001,PRECONDITION,[S3[17:18]::violating || PRECONDITION || S3[15...,[controlled_relation:PRECONDITION]
2,cand_r_00002,FALLING_ACTION,[S3[15:16]::arrest || FALLING_ACTION || S3[24:...,[controlled_relation:FALLING_ACTION]
3,cand_r_00003,FALLING_ACTION,[S3[15:16]::arrest || FALLING_ACTION || S3[31:...,[controlled_relation:FALLING_ACTION]
4,cand_r_00004,FALLING_ACTION,[S3[15:16]::arrest || FALLING_ACTION || S3[39:...,[controlled_relation:FALLING_ACTION]
5,cand_r_00005,PRECONDITION,[S3[43:44]::driving || PRECONDITION || S3[15:1...,[controlled_relation:PRECONDITION]
6,cand_r_00006,PRECONDITION,[S3[44:45]::conviction || PRECONDITION || S3[2...,[controlled_relation:PRECONDITION]
7,cand_r_00007,PRECONDITION,[S3[31:32]::begin || PRECONDITION || S3[39:40]...,[controlled_relation:PRECONDITION]
8,cand_r_00008,PRECONDITION,[S4[15:16]::violation || PRECONDITION || S4[21...,[controlled_relation:PRECONDITION]
9,cand_r_00009,PRECONDITION,[S5[9:12]::rear - ended || PRECONDITION || S5[...,[controlled_relation:PRECONDITION]


Layer 4 assertions


,source,predicate,target,confidence
0,S5[36:37]::driving,PRECONDITION,S3[11:12]::facing,1.0
1,S3[17:18]::violating,PRECONDITION,S3[15:16]::arrest,1.0
2,S3[15:16]::arrest,FALLING_ACTION,S3[24:26]::checked into,1.0
3,S3[15:16]::arrest,FALLING_ACTION,S3[31:32]::begin,1.0
4,S3[15:16]::arrest,FALLING_ACTION,S3[39:40]::stay,1.0
5,S3[43:44]::driving,PRECONDITION,S3[15:16]::arrest,1.0
6,S3[44:45]::conviction,PRECONDITION,S3[24:26]::checked into,1.0
7,S3[31:32]::begin,PRECONDITION,S3[39:40]::stay,1.0
8,S4[15:16]::violation,PRECONDITION,S4[21:22]::case,1.0
9,S5[9:12]::rear - ended,PRECONDITION,S5[6:7]::comes,1.0


Layer 5 triples


,subject,predicate,object,confidence
0,S5[36:37]::driving,PRECONDITION,S3[11:12]::facing,1.0
1,S3[17:18]::violating,PRECONDITION,S3[15:16]::arrest,1.0
2,S3[15:16]::arrest,FALLING_ACTION,S3[24:26]::checked into,1.0
3,S3[15:16]::arrest,FALLING_ACTION,S3[31:32]::begin,1.0
4,S3[15:16]::arrest,FALLING_ACTION,S3[39:40]::stay,1.0
5,S3[43:44]::driving,PRECONDITION,S3[15:16]::arrest,1.0
6,S3[44:45]::conviction,PRECONDITION,S3[24:26]::checked into,1.0
7,S3[31:32]::begin,PRECONDITION,S3[39:40]::stay,1.0
8,S4[15:16]::violation,PRECONDITION,S4[21:22]::case,1.0
9,S5[9:12]::rear - ended,PRECONDITION,S5[6:7]::comes,1.0


Layer 6 ontology relation candidates: 2
Layer 11 completion candidates: 0


## 9. Exact-span-only event projection audit


In [15]:
projection = pd.read_csv(RUN_DIR / "analysis/event_projection_audit.csv")
display(projection)

# Exact-span demonstration using source structure. Gold is consulted only here,
# after the pipeline has completed. Trigger-only fallbacks are disabled.
first_gold_key = next(iter(gold_event_index(gold)["keys_by_id"].values()))[0]
print(first_gold_key)
print(project_event_label(first_gold_key, gold))


,event_id,method,label,candidate_event_ids
0,EVENT_34cccabacad4dea93d3e762114dc05cd,exact_sentence_token_span,S1[2:4]::checks into,NaN
1,NaN,unmapped_or_ambiguous_exact_span,S3[1:3]::skipping out,[]
2,EVENT_b79df11f8737f700691af4f8a7132190,exact_sentence_token_span,S3[4:5]::entering,NaN
3,NaN,unmapped_or_ambiguous_exact_span,S3[11:12]::facing,[]
4,EVENT_c332a6b60af0495f34856f112dc68632,exact_sentence_token_span,S3[15:16]::arrest,NaN
5,EVENT_9c07ef70f303f23dd82b32fb28a61ecb,exact_sentence_token_span,S3[17:18]::violating,NaN
6,EVENT_d7aec1e4ff20c3264f9fb7686355eb8a,exact_sentence_token_span,S3[24:26]::checked into,NaN
7,NaN,unmapped_or_ambiguous_exact_span,S3[31:32]::begin,[]
8,EVENT_2ea9a901cc230afcc371bcffee02a551,exact_sentence_token_span,S3[39:40]::stay,NaN
9,NaN,unmapped_or_ambiguous_exact_span,S3[43:44]::driving,[]


S4[15:16]::violation
{'event_id': 'EVENT_434faad0c9ec71baa6a91df3c385299b', 'method': 'exact_sentence_token_span', 'label': 'S4[15:16]::violation'}


## 10. Speed, concurrency and errors

In [16]:
def optional_jsonl(path: Path) -> pd.DataFrame:
    return pd.DataFrame(read_jsonl(path)) if path.is_file() else pd.DataFrame()

calls = optional_jsonl(RUN_DIR / "run_logs/llm_calls.jsonl")
errors = optional_jsonl(RUN_DIR / "run_logs/llm_errors.jsonl")
parse_errors = optional_jsonl(RUN_DIR / "run_logs/llm_parse_errors.jsonl")
retrieval = optional_jsonl(RUN_DIR / "run_logs/ontology_retrieval.jsonl")

if not calls.empty:
    display(calls)
    display(calls.groupby("layer_tag").agg(
        calls=("call_index", "count"),
        total_recorded_seconds=("elapsed_seconds", "sum"),
        maximum_call_seconds=("elapsed_seconds", "max"),
        mean_system_chars=("system_chars", "mean"),
        mean_user_chars=("user_chars", "mean"),
        mean_response_chars=("response_chars", "mean"),
    ).reset_index())

print("Backend/API errors:", len(errors))
if not errors.empty: display(errors)
print("JSON parse errors:", len(parse_errors))
if not parse_errors.empty: display(parse_errors)
if not retrieval.empty:
    display(retrieval.groupby("layer_name").size().reset_index(name="retrieval_calls"))

manifest = read_json(RUN_DIR / "run_manifest.json")
print("Wall-clock pipeline seconds:", manifest.get("elapsed_seconds"))

,call_index,layer_tag,model,temperature,message_count,system_chars,user_chars,max_tokens,request_timeout,started_at,status,elapsed_seconds,response_chars,response_path,json_parse_ok,parsed_type,parse_error
0,3,layer01_event_inventory_v1_5,openai/gpt-oss-20b,0.0,2,1675,2500,8192,180,2026-08-01 15:55:42,ok,2.250,337,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
1,1,layer01_event_inventory_v1_5,openai/gpt-oss-20b,0.0,2,1675,2281,8192,180,2026-08-01 15:55:42,ok,2.267,206,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
2,4,layer01_event_inventory_v1_5,openai/gpt-oss-20b,0.0,2,1675,2651,8192,180,2026-08-01 15:55:42,ok,4.267,738,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
3,2,layer01_event_inventory_v1_5,openai/gpt-oss-20b,0.0,2,1675,2841,8192,180,2026-08-01 15:55:42,ok,378.549,725,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
4,5,layer01_event_inventory_v1_5,openai/gpt-oss-20b,0.0,2,986,5742,8192,180,2026-08-01 16:02:01,ok,9.788,208,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
5,6,layer01_event_inventory_v1_5,openai/gpt-oss-20b,0.0,2,1016,5742,8192,180,2026-08-01 16:02:10,ok,7.766,208,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
6,7,layer01_event_inventory_v1_5,openai/gpt-oss-20b,0.0,2,1027,5742,8192,180,2026-08-01 16:02:18,ok,2.470,349,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
7,8,layer01_event_inventory_v1_5,openai/gpt-oss-20b,0.0,2,1098,5012,8192,180,2026-08-01 16:02:21,ok,7.675,1020,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
8,10,layer02_batched_five_way_relations_v1_5,openai/gpt-oss-20b,0.0,2,1910,36522,4096,180,2026-08-01 16:02:28,ok,4.113,1350,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None
9,11,layer02_batched_five_way_relations_v1_5,openai/gpt-oss-20b,0.0,2,1910,26955,4096,180,2026-08-01 16:02:28,ok,5.712,1810,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...,True,dict,None


,layer_tag,calls,total_recorded_seconds,maximum_call_seconds,mean_system_chars,mean_user_chars,mean_response_chars
0,layer01_event_inventory_v1_5,8,415.032,378.549,1353.375,4063.875,473.875
1,layer02_batched_five_way_relations_v1_5,35,399.354,44.714,1615.000,26108.400,1574.600


Backend/API errors: 0
JSON parse errors: 0
Wall-clock pipeline seconds: 541.2258195877075


## Success checklist before the five-document batch

1. The sibling RAGTree OWL-Time file resolves and remains secondary evidence.
2. Three coverage reviews materially improve exact event-span recall, including repeated mentions and eventive nouns.
3. Layer 1 deterministically builds the full unordered pair pool for this small document with **zero** relation-generation LLM calls.
4. Layer 2 classifies bounded pair batches with exactly five decisions, both directions considered, textual evidence required, one recovery pass, and cached responses.
5. `NONE`, conflicting, malformed, and evidence-free outputs are absent from Layer 3 relation candidates.
6. The candidate-pool recall given available endpoints is high; missed relations are separated into event, pool, `NONE`, direction, class, and Layer 4/5 failures.
7. Exact-span-only projected metrics and strict native span metrics are both reported; no global trigger fallback remains.
8. Gold remains unavailable until post-Layer-12 evaluation. Freeze v1.5 only after this one-document result is satisfactory.
